# HomeostaticDysregulation

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.HomeostaticDysregulation)

class HomeostaticDysregulation(pyagingModel):
    """Mahalanobis distance from a young, healthy NHANES III reference cohort."""

    def __init__(self):
        super().__init__()
        for sex in ["male", "female"]:
            for name in ["reference_mean", "reference_sd", "center", "precision", "log_hd_sd"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein."""
        return log1p_crp(self.features, x)

    def postprocess(self, x):
        """Score the log Mahalanobis distance from the sex's reference cohort.

        Notes
        -----
        ``center`` is not zero. ``hd_calc`` takes each column's mean and standard
        deviation with ``na.rm = TRUE`` over the full reference column and drops
        incomplete rows only afterwards, so the surviving rows' mean is not the
        centring constant; the residual offset reaches 0.19 standard deviations
        and 

In [3]:
model = pya.models.HomeostaticDysregulation()

## Define clock metadata

Each `# Paper:` comment reproduces the evidence recorded for that field in `clocks/metadata/evidence_ledger.jsonl`; `validate_metadata.py` compares the two, so they cannot drift apart. The paper is not open access, so the fields describing how the reference cohort was assembled and how the score is formed are sourced from the BioAge package code, which is also where the parameters themselves come from.

In [4]:
model.metadata["clock_name"] = 'homeostaticdysregulation'
model.metadata["data_type"] = 'clinical biomarkers'  # Paper: blood chemistry and organ function test data
model.metadata["species"] = 'Homo sapiens'  # Paper: Homo sapiens
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.'
model.metadata["doi"] = 'https://doi.org/10.1007/s11357-021-00480-5'
model.metadata["notes"] = "Mahalanobis distance from a young, healthy NHANES III reference cohort, fit separately by sex. The output is a log dysregulation score and is NOT expressed in years; larger values mean greater dysregulation. BioAge divides the score by the projection cohort's standard deviation of log(distance), so that NHANES IV constant is baked in per sex to keep single-sample predictions reproducible. C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male reference. An incomplete panel biases the score downward: the score is a distance, so filling an absent biomarker with its reference value removes that marker's own contribution, which lowered the score for every one of the nine markers on average across the reference subjects and by as much as 2.9 on a 1.98-6.76 range. An incomplete panel therefore reads as healthier than it is, so the missing-feature warning the prediction pipeline emits should be heeded."
model.metadata["research_only"] = None
model.metadata["tissue"] = ['blood']  # Paper: blood chemistry
model.metadata["predicts"] = ['biological age']  # Paper: implements three published methods to quantify biological aging based on analysis of chronological age and mortality risk: Klemera-Doubal biological age, PhenoAge, and homeostatic dysregulation
model.metadata["training_target"] = ['not applicable']  # Paper: means = colMeans(ref); cv_mat = var(ref)
model.metadata["unit"] = ['unitless']  # Paper: dat$hd_log = log(hd)/sd(log(hd))
model.metadata["model_type"] = 'Mahalanobis distance composite'  # Paper: hd[x] <- sqrt((dat[x, ] - means) %*% solve(cv_mat) %*% (dat[x, ] - means))
model.metadata["platform"] = ['clinical laboratory assays']  # Paper: blood chemistry and organ function test data
model.metadata["population"] = 'adults'  # Paper: train = NHANES3 %>% filter(age >= 20 & age <= 30 & pregnant == 0 & bmi < 30)
model.metadata["journal"] = 'GeroScience'
model.metadata["last_author"] = 'Daniel W. Belsky'
model.metadata["n_features"] = 10
model.metadata["citations"] = 332
model.metadata["citations_date"] = '2026-08-20'

## Download clock dependencies

The fitted parameters were extracted from the R package [dayoonkwon/BioAge](https://github.com/dayoonkwon/BioAge) and are checked in under `clocks/bioage_params/homeostaticdysregulation.json`. Each sex has, per biomarker, the reference cohort's `reference_mean` and `reference_sd`, plus the `standardized_center` and `standardized_covariance` of the standardized reference matrix and the scalar `log_hd_sd`.

In [5]:
with open("../bioage_params/homeostaticdysregulation.json") as handle:
    params = json.load(handle)

params["features"]

['albumin',
 'lymphocyte_percent',
 'mean_cell_volume',
 'glucose',
 'red_cell_distribution_width',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'white_blood_cell_count',
 'female']

## Load features

In [6]:
model.features = params["features"]
model.features

['albumin',
 'lymphocyte_percent',
 'mean_cell_volume',
 'glucose',
 'red_cell_distribution_width',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'white_blood_cell_count',
 'female']

#### Normal feature ranges

Every feature above is registered in `pyaging`'s feature range registry, which is the single source of truth for units and plausible bounds; the clock stores those units in `model.feature_units`, so a saved clock is self-describing and carries its own units even if the package registry later standardizes differently. `tests/test_clock_metadata.py` asserts every built clock's stored copy still matches the registry, so a registry correction cannot be silently shadowed by a stale one. `pya.utils.get_feature_ranges("homeostaticdysregulation")` reports them for a saved clock.

The reference cohort was fit on BioAge's SI-unit variants, so no post-hoc unit conversion is applied anywhere in this clock. As in `kdmage`, `c_reactive_protein` is supplied **raw, in mg/dL**: BioAge's `lncrp` biomarker is `log1p(CRP in mg/dL)` — not the natural log that `phenoage` uses — so `HomeostaticDysregulation.preprocess` applies `log1p` internally and the reference mean, SD, centre, and covariance for that biomarker stay on the `log1p` scale. A raw CRP is clamped to 0.01 mg/dL first, which is the registered lower bound, so a below-detection reading coded as `0` or an absent column cannot send `-inf` into the distance.

`female` is coded 1 = female, 0 = male, and it selects which sex's reference the sample is scored against. A dataset with no `female` column therefore scores every sample against the male reference.

In [ ]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

## Load weights into base model

There is nothing to learn here either: the homeostatic dysregulation score is a Mahalanobis distance against a fixed reference cohort, so the base model is the identity and all of the arithmetic lives in `HomeostaticDysregulation.postprocess`. The parameters are stored as buffers, one set per sex, with the covariance inverted once at build time.

Two subtleties are worth stating explicitly, because both are easy to get silently wrong:

- **`standardized_center` is not zero.** `hd_calc` computes each column's `mean` and `sd` with `na.rm = TRUE` over the *full* reference column and only afterwards drops rows with any missing marker, so the surviving rows' mean is not the centring constant. The residual offset reaches 0.186 standard deviations (male red cell distribution width). Dropping it shifts the score by up to 0.067 on values spanning 1.98-6.76 — roughly 1-3%, in a subject-varying direction that does not cancel.
- **`log_hd_sd` is a cohort constant.** BioAge reports `hd_log = log(hd)/sd(log(hd))`, where the standard deviation runs over whatever cohort is being scored. Baking in the NHANES IV projection cohort's value is what makes a single-sample prediction well defined at all; it also means this clock's scores are only comparable to BioAge's when BioAge is run on that cohort.

`params[sex]["biomarkers"]` need not be in `model.features` order, so every vector is reindexed onto the feature order before it is stored, and the covariance is permuted on both axes.

In [8]:
model.base_model = torch.nn.Identity()

for sex in ["male", "female"]:
    fit = params[sex]
    order = [fit["biomarkers"].index(name) for name in model.features[:-1]]
    for key, buffer in [
        ("reference_mean", "reference_mean"),
        ("reference_sd", "reference_sd"),
        ("standardized_center", "center"),
    ]:
        setattr(model, f"{buffer}_{sex}", torch.tensor([fit[key][index] for index in order], dtype=torch.float64))
    covariance = torch.tensor(fit["standardized_covariance"], dtype=torch.float64)[order][:, order]
    setattr(model, f"precision_{sex}", torch.linalg.inv(covariance))
    setattr(model, f"log_hd_sd_{sex}", torch.tensor(fit["log_hd_sd"], dtype=torch.float64))

    # The reindex above is a no-op whenever the two orders already agree, which is exactly
    # when a mangled copy of it would go unnoticed. Check it mapped what it claims.
    for position, name in enumerate(model.features[:-1]):
        source = fit["biomarkers"].index(name)
        for key, buffer in [
            ("reference_mean", "reference_mean"),
            ("reference_sd", "reference_sd"),
            ("standardized_center", "center"),
        ]:
            assert getattr(model, f"{buffer}_{sex}")[position].item() == fit[key][source], (sex, key, name)

model.center_male

tensor([ 0.1358,  0.0026, -0.0385,  0.0287, -0.1860, -0.0952, -0.0938, -0.0040,
        -0.0027], dtype=torch.float64)

## Load reference values

`check_features_in_adata` substitutes these for any feature a user's dataframe does not carry. Filling an absent biomarker with `0` would be badly wrong here: a zero assay sits 5 to 25 reference standard deviations from the centre, and because the score is a distance, that one column would dominate it outright.

The right fill is the point where the biomarker's own standardized, centred deviation is exactly zero, so it adds nothing of its own to the distance: `reference_mean + standardized_center * reference_sd`. That is the reference distribution's centre, not its raw mean — the two differ precisely because of the `na.rm` subtlety noted above.

The centre is sex-specific but `reference_values` is a single vector, and the two sexes' centres are stated in raw units on different scales, so averaging them in raw units is not the right compromise. What matters is the residual measured in standard deviations, because that is what the distance sees. We therefore pick the value whose standardized residual is equal and opposite for the two sexes,

    (a_male * sd_female + a_female * sd_male) / (sd_male + sd_female),

which minimises the worst-case residual across the sexes. For seven of the nine biomarkers that residual is under 0.55 standard deviations. Creatinine is the outlier at 1.22, and irreducibly so: the male and female reference centres are 2.4 standard deviations apart, so no single value can be neutral for both. Albumin and alkaline phosphatase are next at about 0.5. `tests/predict/test_bioage_clocks.py` pins the bound and the equal-and-opposite construction.

Measured over the 20 reference subjects, dropping one biomarker moves the score at most **2.92** from its complete-data value when reference-filled, against **4.66** when zero-filled, on scores that span 1.98-6.76. C-reactive protein is the one biomarker where zero-filling is not much worse (0.58 against 1.02): `log1p` compresses a clamped zero to only 1.8 standard deviations below a reference centre that is itself near the floor, because the reference cohort was screened to `crp < 2`. Everywhere else reference-filling wins by a factor of 3 to 18.

The CRP slot needs `expm1`: reference values are substituted into the *input*, so they are in user-facing units, and `preprocess` will apply `log1p` to whatever sits there.

`female` is set to `0`, which is not a neutral choice: it means a dataset with no sex column is scored entirely against the male reference. That is recorded in the metadata notes.

In [9]:
crp = model.features.index("c_reactive_protein")
centre = {
    sex: getattr(model, f"reference_mean_{sex}") + getattr(model, f"center_{sex}") * getattr(model, f"reference_sd_{sex}")
    for sex in ["male", "female"]
}
sd = {sex: getattr(model, f"reference_sd_{sex}") for sex in ["male", "female"]}
reference = (
    (centre["male"] * sd["female"] + centre["female"] * sd["male"]) / (sd["male"] + sd["female"])
).tolist()
reference[crp] = math.expm1(reference[crp])  # stored raw; preprocess applies log1p

model.reference_values = reference + [0.0]  # female: no sex column means the male reference

assert len(model.reference_values) == len(model.features)

# The residual the compromise leaves, in reference standard deviations.
standardized = torch.tensor(reference, dtype=torch.float64)
standardized[crp] = math.log1p(standardized[crp])
pd.DataFrame(
    {
        "feature": model.features[:-1],
        **{sex: ((standardized - centre[sex]) / sd[sex]).tolist() for sex in ["male", "female"]},
    }
)

,feature,male,female
0,albumin,-0.490255,0.490255
1,lymphocyte_percent,0.011576,-0.011576
2,mean_cell_volume,-0.061514,0.061514
3,glucose,-0.260534,0.260534
4,red_cell_distribution_width,0.097374,-0.097374
5,creatinine,-1.222243,1.222243
6,c_reactive_protein,0.160685,-0.160685
7,alkaline_phosphatase,-0.513524,0.513524
8,white_blood_cell_count,0.057104,-0.057104


## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "log_mahalanobis_distance"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for '
             'quantification of biological age from blood chemistry and organ '
             'function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'citations': 332,
 'citations_date': '2026-08-20',
 'clock_name': 'homeostaticdysregulation',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'journal': 'GeroScience',
 'last_author': 'Daniel W. Belsky',
 'model_type': 'Mahalanobis distance composite',
 'n_features': 10,
 'notes': 'Mahalanobis distance from a young, healthy NHANES III reference '
          'cohort, fit separately by sex. The output is a log dysregulation '
          'score and is NOT expressed in years; larger values mean greater '
          "dysregulation. BioAge divides the score by the pr

## Basic test

The smoke test feeds the midpoint of each feature's registered range, so the inputs are physiologically plausible rather than random.

In [13]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[11.4022]], dtype=torch.float64)

#### Parity with BioAge

The acceptance gate is `tests/predict/test_bioage_clocks.py`, which reproduces BioAge's own output for 20 NHANES IV subjects. Reproduced inline here as well, since a notebook that builds parameters should show that they land where the source package lands.

In [14]:
with open("../bioage_params/reference_predictions.json") as handle:
    reference_predictions = json.load(handle)

matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference_predictions["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference_predictions["expected"]["homeostaticdysregulation"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 8.455458555545192e-13


## Save torch model

In [15]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [16]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)